# Homodyne $J_z$ - Generic $N$ atoms (focus: $N=3$)

Notebook generico per adattare il problema a $N$ atomi:
- stati iniziali `plusN` e `cssN`
- canali di collasso costruiti come combinazioni lineari delle singole $\sigma_z^{(i)}$
  tramite una matrice unitaria $U$ (`DFT_N`, `tritter` 3x3, `collective_diff_3`)
- simulazione omodina (`smesolve`)
- osservabili: **concurrence media pairwise** e **spin squeezing KU**.


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import importlib
import quantum_function2
importlib.reload(quantum_function2)
from quantum_function2 import save_ineff_df_npz, angle_to_path

from itertools import combinations
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qutip import (
    basis,
    concurrence,
    expect,
    ket2dm,
    qeye,
    sigmaz,
    sigmax,
    sigmay,
    smesolve,
    tensor,
)


In [ ]:
# --- SAFETY SWITCH ---
RUN_THIS_CELL = True

if not RUN_THIS_CELL:
    raise SystemExit("Cell locked. Set RUN_THIS_CELL=True to run")

plt.rcParams["text.usetex"] = True
plt.rcParams.update({
    "mathtext.fontset": "cm",
    "font.family": "serif",
    "font.size": 14,
    "axes.unicode_minus": False,
})


In [ ]:
# DEFINITIONS 

def single_site_op(op, site: int, N: int):
    ops = [qeye(2) for _ in range(N)]
    ops[site] = op
    return tensor(ops)


def local_sigma_z_ops(N: int):
    sz = sigmaz()
    return [single_site_op(sz, i, N) for i in range(N)]


def collective_spin_ops(N: int):
    Jx = 0
    Jy = 0
    Jz = 0
    for i in range(N):
        Jx += 0.5 * single_site_op(sigmax(), i, N)
        Jy += 0.5 * single_site_op(sigmay(), i, N)
        Jz += 0.5 * single_site_op(sigmaz(), i, N)
    return Jx, Jy, Jz


# ---------- Initial states ----------

def state_plus_N(N: int):
    g = basis(2, 0)
    e = basis(2, 1)
    plus = (g + e).unit()
    return tensor([plus for _ in range(N)])


def state_css_N(N: int, theta: float, phi: float):
    # Same convention used in your N=2 helper.
    single = (np.sin(theta / 2.0) * basis(2, 0) + np.exp(1j * phi) * np.cos(theta / 2.0) * basis(2, 1)).unit()
    return tensor([single for _ in range(N)])


# ---------- Interferometer unitaries ----------

def dft_unitary(N: int):
    j = np.arange(N)
    k = np.arange(N)
    J, K = np.meshgrid(j, k, indexing="ij")
    return np.exp(2j * np.pi * J * K / N) / np.sqrt(N)


def tritter_unitary_3():
    # Standard 3x3 DFT tritter.
    return dft_unitary(3)


def collective_diff_unitary_3():
    # row0: collective sum mode
    # row1,row2: orthonormal difference modes
    return np.array([
        [1/np.sqrt(3),  1/np.sqrt(3),  1/np.sqrt(3)],
        [1/np.sqrt(2), -1/np.sqrt(2),  0],
        [1/np.sqrt(6),  1/np.sqrt(6), -2/np.sqrt(6)],
    ], dtype=complex)


def choose_unitary(N: int, mode: str):
    if mode == "dft":
        return dft_unitary(N)
    if mode == "tritter":
        if N != 3:
            raise ValueError("'tritter' is defined only for N=3")
        return tritter_unitary_3()
    if mode == "collective_diff_3":
        if N != 3:
            raise ValueError("'collective_diff_3' is defined only for N=3")
        return collective_diff_unitary_3()
    raise ValueError(f"Unknown unitary mode: {mode}")


def unitary_error(U: np.ndarray):
    N = U.shape[0]
    return np.linalg.norm(U.conj().T @ U - np.eye(N))


# ---------- Collapsing operators ----------

def build_collapsing_ops_sigma_z(N: int, gamma: float, U: np.ndarray):
    if U.shape != (N, N):
        raise ValueError(f"U must be {N}x{N}, got {U.shape}")
    local_ops = local_sigma_z_ops(N)
    c_ops = []
    for k in range(N):
        ck = 0
        for j in range(N):
            ck += U[k, j] * local_ops[j]
        c_ops.append(np.sqrt(gamma) * ck)
    return c_ops


def split_observed_unobserved(collapse_ops: list, eta):
    # eta: scalar or list/array length N
    N = len(collapse_ops)
    eta_vec = np.asarray(eta, dtype=float)
    if eta_vec.ndim == 0:
        eta_vec = np.full(N, float(eta_vec))
    if eta_vec.shape[0] != N:
        raise ValueError(f"eta must be scalar or length {N}, got shape {eta_vec.shape}")
    if np.any(eta_vec < 0) or np.any(eta_vec > 1):
        raise ValueError("All eta values must be in [0,1]")

    c_unobs = [np.sqrt(1.0 - eta_vec[k]) * collapse_ops[k] for k in range(N)]
    c_obs = [np.sqrt(eta_vec[k]) * collapse_ops[k] for k in range(N)]
    return c_unobs, c_obs, eta_vec


# ---------- Observables ----------

def concurrence_pairwise_mean(state, N: int):
    if N < 2:
        return 0.0
    rho = ket2dm(state) if state.isket else state
    vals = []
    for i, j in combinations(range(N), 2):
        rho_ij = rho.ptrace([i, j])
        vals.append(float(np.real_if_close(concurrence(rho_ij))))
    return float(np.mean(vals))


def xi_ku_general(state, Jx, Jy, Jz, N: int, tol: float = 1e-12):
    Jx_exp = float(np.real_if_close(expect(Jx, state)))
    Jy_exp = float(np.real_if_close(expect(Jy, state)))
    Jz_exp = float(np.real_if_close(expect(Jz, state)))
    Jmean = np.array([Jx_exp, Jy_exp, Jz_exp], dtype=float)
    m = np.linalg.norm(Jmean)

    Js = [Jx, Jy, Jz]
    C = np.zeros((3, 3), dtype=float)
    for i, Ji in enumerate(Js):
        for j, Jj in enumerate(Js):
            C[i, j] = float(np.real_if_close(0.5 * expect(Ji * Jj + Jj * Ji, state) - Jmean[i] * Jmean[j]))

    if m > tol:
        u = Jmean / m
        a = np.array([1.0, 0.0, 0.0]) if abs(u[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
        e1 = a - u * np.dot(u, a)
        e1 /= np.linalg.norm(e1)
        e2 = np.cross(u, e1)

        C2 = np.array([
            [e1 @ C @ e1, e1 @ C @ e2],
            [e2 @ C @ e1, e2 @ C @ e2],
        ], dtype=float)
        lam_min = np.linalg.eigvalsh(C2)[0]
    else:
        lam_min = np.linalg.eigvalsh(C)[0]

    lam_min = max(lam_min, 0.0)
    return float((4.0 / N) * lam_min)


def make_observables(N: int):
    Jx, Jy, Jz = collective_spin_ops(N)

    def conc_eop(_t, state):
        return concurrence_pairwise_mean(state, N)

    def xi_eop(_t, state):
        return xi_ku_general(state, Jx, Jy, Jz, N)

    return [conc_eop, xi_eop]


def normalize_expect_chunk(sol_expect, n_eops: int, n_times: int):
    arr = np.asarray(sol_expect)
    arr = np.real_if_close(arr)
    arr = np.asarray(arr, dtype=float)

    # common shape with keep_runs_results=False
    if arr.ndim == 2 and arr.shape == (n_eops, n_times):
        return arr

    # if per-run is returned, average over traj axis
    if arr.ndim == 3:
        # (n_eops, ntraj, n_times)
        if arr.shape[0] == n_eops and arr.shape[2] == n_times:
            return arr.mean(axis=1)
        # (ntraj, n_eops, n_times)
        if arr.shape[1] == n_eops and arr.shape[2] == n_times:
            return arr.mean(axis=0)

    if n_eops == 1 and arr.ndim == 1 and arr.shape[0] == n_times:
        return arr.reshape(1, -1)

    raise ValueError(f"Unexpected expect shape {arr.shape}")


def run_homodyne_avg_N(
    rho0,
    times,
    H,
    collapse_ops,
    eta,
    e_ops,
    ntraj: int,
    chunk_size: int,
    num_cpus: int,
    seed: int,
):
    options = {
        "keep_runs_results": False,
        "num_cpus": max(1, num_cpus - 1),
        "map": "parallel" if num_cpus > 1 else "serial",
    }

    n_eops = len(e_ops)
    n_times = len(times)

    weighted = None
    done = 0
    chunk_id = 0
    rng = np.random.default_rng(seed)

    c_unobs, c_obs, eta_vec = split_observed_unobserved(collapse_ops, eta)

    while done < ntraj:
        chunk_id += 1
        n_this = min(chunk_size, ntraj - done)
        np.random.seed(int(rng.integers(0, 2**31 - 1)))

        sol = smesolve(
            H,
            rho0,
            times,
            c_ops=c_unobs,
            sc_ops=c_obs,
            heterodyne=False,
            e_ops=e_ops,
            ntraj=n_this,
            options=options,
        )

        mean_chunk = normalize_expect_chunk(sol.expect, n_eops=n_eops, n_times=n_times)

        if weighted is None:
            weighted = mean_chunk * n_this
        else:
            weighted += mean_chunk * n_this

        done += n_this
        print(f"chunk {chunk_id}: {done}/{ntraj} trajectories")

    return weighted / float(ntraj), eta_vec


In [ ]:
# UNITARY QUICK CHECK (optional)
U_dft3 = dft_unitary(3)
U_tri = tritter_unitary_3()
U_coldiff = collective_diff_unitary_3()

print("||U_dft3^dag U_dft3 - I|| =", unitary_error(U_dft3))
print("||U_tritter^dag U_tritter - I|| =", unitary_error(U_tri))
print("||U_coldiff^dag U_coldiff - I|| =", unitary_error(U_coldiff))

print("\nU collective_diff_3 =")
print(np.round(U_coldiff, 6))


In [ ]:
# PARAMETERS
N = 3                         # set N generic
unitary_mode = "collective_diff_3"   # "dft", "tritter", "collective_diff_3"

gamma = 1.0
omega = 0.0                   # H = (omega/2) sum sigma_z^(i)

# fixed interferometer phases are inside U; no explicit phi1/phi2 here for generic N

# channel efficiencies (scalar or vector length N)
# examples:
# eta = 1.0
# eta = np.linspace(1.0, 0.5, N)
eta = 1.0

T1 = 1.0 / gamma
t_end = 10.0
dt = 0.02
times = np.arange(0.0, t_end * T1, dt * T1)

ntraj = 400
chunk_size = 100
num_cpus = max(1, os.cpu_count() - 1)
seed = 12345

# initial states to compare
initial_states = {
    "plusN": {"kind": "plus"},
    "css_pi_3": {"kind": "css", "theta": np.pi / 3.0, "phi": 0.0},
}

out_root = Path(r".\Graphs\Jz_N_Homodyne_Generic")

print(f"N={N}, unitary_mode={unitary_mode}, ntraj={ntraj}")


In [ ]:
# BUILD MODEL OBJECTS
U = choose_unitary(N, unitary_mode)
err = unitary_error(U)
print(f"unitarity error = {err:.3e}")

collapse_ops = build_collapsing_ops_sigma_z(N=N, gamma=gamma, U=U)

# Hamiltonian
H = 0
for i in range(N):
    H += 0.5 * omega * single_site_op(sigmaz(), i, N)

e_ops = make_observables(N)
columns = ["Conc_pair_mean", "Xi2_KU"]

# visualize first mode structure
print("first output mode coefficients (row 0 of U):")
print(np.round(U[0], 6))


In [ ]:
# SIMULATION
results = {}
eta_used = None

for label, cfg in initial_states.items():
    if cfg["kind"] == "plus":
        psi0 = state_plus_N(N)
    elif cfg["kind"] == "css":
        psi0 = state_css_N(N, theta=cfg["theta"], phi=cfg["phi"])
    else:
        raise ValueError(f"Unknown state kind in {label}: {cfg['kind']}")

    rho0 = ket2dm(psi0)

    mean_arr, eta_vec = run_homodyne_avg_N(
        rho0=rho0,
        times=times,
        H=H,
        collapse_ops=collapse_ops,
        eta=eta,
        e_ops=e_ops,
        ntraj=ntraj,
        chunk_size=chunk_size,
        num_cpus=num_cpus,
        seed=seed + hash(label) % 100000,
    )

    eta_used = eta_vec
    df = pd.DataFrame(mean_arr.T, columns=columns)
    df.insert(0, "step", np.arange(len(times), dtype=int))
    df.insert(1, "t", times)
    df.insert(2, "t_T1", times / T1)
    results[label] = df

print("Done")
print("eta vector used:", np.round(eta_used, 4))
results[list(results.keys())[0]].head()


In [ ]:
# PLOT
x = times / T1

for col in columns:
    plt.figure(figsize=(12, 8))
    for label, df in results.items():
        plt.plot(x, df[col].to_numpy(), linewidth=2.0, label=label)

    if col == "Xi2_KU":
        plt.axhline(1.0, color="k", linestyle="--", linewidth=1.2, alpha=0.6)

    plt.xlim(0.0, t_end)
    plt.xlabel(r"$t/T_1$")
    plt.ylabel(r"$\overline{\mathcal{C}}_{\mathrm{pair}}$" if col == "Conc_pair_mean" else r"$\overline{\xi^2_{KU}}$")
    plt.title(rf"Homodyne Jz generic N={N}, U={unitary_mode}, ntraj={ntraj}")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(loc="best")
    plt.show()


In [ ]:
# SAVE (mean curves + metadata)
unitary_tag = unitary_mode
out_dir = out_root / f"N={N}_U={unitary_tag}"
out_dir.mkdir(parents=True, exist_ok=True)

# save each state dataframe as CSV
for label, df in results.items():
    df.to_csv(out_dir / f"mean_observables_{label}.csv", index=False)

# also save as compressed NPZ using thesis helper
save_dict = {label: df.copy() for label, df in results.items()}
npz_path = out_dir / "mean_observables_all_states.npz"
save_ineff_df_npz(save_dict, npz_path, meta=None, step_col="step")

meta = {
    "N": N,
    "measurement": "homodyne",
    "gamma": gamma,
    "omega": omega,
    "ntraj": ntraj,
    "chunk_size": chunk_size,
    "dt_T1": dt,
    "t_end_T1": t_end,
    "unitary_mode": unitary_mode,
    "unitary_error": float(err),
    "eta_vector": [float(x) for x in eta_used],
    "initial_states": initial_states,
    "columns": columns,
    "U_real": U.real.tolist(),
    "U_imag": U.imag.tolist(),
}
(out_dir / "run_config.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print(f"Saved in: {out_dir}")
